# Electron-electron covariance on a kinetic-energy axis — scaled partial-covariance test

**Task.** Plot the config-1 electron-electron covariance `Cov(D, D)` on an
electron kinetic-energy axis, using the TOF -> KE calibration produced by
Task 1 (`etof_energy_calibration.ipynb`, Argon run 58826 minus residual-gas
run 58825), and test how *scaling* the GMD-driven common-mode subtraction
changes the shape of the covariance map.

The standard partial covariance removes the rank-1 pulse-energy term with a
fixed unit weight:

    pCov(alpha) = Cov(D, D) - alpha * Corr,        Corr = Cov(D, G) Cov(D, G)^T / Var(G)

with `alpha = 1`. Here `alpha` is instead treated as a free scaling factor,
chosen to minimise the residual covariance inside a region of the map that is
**a priori uncorrelated** — a block of (KE_a, KE_b) pairs where no physical
electron-electron coincidence is expected, so anything surviving there is
uncompensated (or over-compensated) common mode. Because `pCov` is linear in
`alpha`, the least-squares minimiser over the region R is closed-form:

    alpha* = sum_R( Cov * Corr ) / sum_R( Corr^2 )

**Data.**
- A config-1 aggregates H5 written by `compute_aggregates.py`
  (`D`/`DtD`/`DtG`/`G`/`GtG` per GMD bin); file set in the parameters cell.
- The persisted calibration `analysis/post/calibration/etof_tof_to_ke_argon_nozzle_in.json`.

**Steps.**
1. Load the aggregates and the TOF -> KE calibration.
2. Form `Cov(D, D)`, `Cov(D, G)`, `Var(G)` per GMD bin and the rank-1 correction.
3. Map the eTOF bin axes onto KE (Jacobian-corrected density for plotting).
4. Define the a-priori-uncorrelated region (KE rectangles, diagonal excluded).
5. Solve for `alpha*` per GMD bin and scan the residual vs `alpha`.
6. Compare maps: raw, standard partial (`alpha = 1`), region-minimising (`alpha*`).

Started from `covariance_tests.ipynb` (config-1 covariance machinery).

In [ ]:
import sys
import json
from pathlib import Path

# analysis/post -> repo root is two levels up (same layout as analysis/notebooks).
_REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(_REPO_ROOT / "analysis" / "scripts"))

import numpy as np
import matplotlib.pyplot as plt

import config
from compute_aggregates import load_aggregates

%matplotlib inline

## Parameters

In [ ]:
# Config-1 aggregates file (D = electron TOF, C = ion TOF, G = GMD).
AGGREGATES_FILE = config.COMBINED_DIR / "glycine_delay_scan_150C_272.0eV_aggregates.h5"

# TOF -> KE calibration persisted by etof_energy_calibration.ipynb (Task 1).
CALIBRATION_JSON = _REPO_ROOT / "analysis" / "post" / "calibration" / "etof_tof_to_ke_argon_nozzle_in.json"

# The aggregates tof_edges are in units of 100 ps (see aggregates_example_cfg1.py),
# while the calibration model works in ns.
TOF_UNIT_NS = 0.1

# KE range shown on the maps (eV). eTOF bins mapping outside this range
# (or with t <= t0, where the model is unphysical) are dropped.
KE_PLOT_RANGE = (5.0, 280.0)

# Plot covariance as a density per eV^2 (divide by the KE widths of both
# axes) so peak areas — not heights — are preserved under the nonlinear
# TOF -> KE mapping.
JACOBIAN_CORRECT = True

# A-priori-uncorrelated region: list of KE rectangle pairs
# ((ke_a_lo, ke_a_hi), (ke_b_lo, ke_b_hi)). Each pair marks the block of
# the map where electrons from range a and range b are NOT expected to be
# physically coincident; the mask is symmetrised automatically.
# EDIT from the raw map (dashed rectangles overlaid below) before
# trusting alpha*.
UNCORRELATED_ROI_KE = [
    ((30.0, 60.0), (150.0, 250.0)),
]

# Exclude the autocorrelation ridge |KE_i - KE_j| < this (eV) from the
# region (the diagonal is always positive: shot-noise variance).
DIAG_EXCLUDE_EV = 5.0

# alpha values scanned for the residual curve. The closed-form alpha* is
# exact; the scan is a visual check that the minimum sits where expected.
ALPHA_SCAN = np.linspace(0.0, 2.0, 201)

# GMD bin used for the detailed single-bin comparison figure.
# -1 = highest-GMD bin (strongest common mode, largest alpha effect).
FOCUS_BIN = -1

CLIM_PERCENTILE = 99.0

## Load aggregates

In [ ]:
agg = load_aggregates(AGGREGATES_FILE)
assert agg.config == 1, "This notebook expects a config-1 aggregates file."

print(f"file        : {AGGREGATES_FILE}")
print(f"mode        : {agg.mode}")
print(f"n_gmd_bins  : {agg.n_gmd_bins}")
print(f"n_tof (e)   : {agg.n_tof}   (edges {agg.tof_edges[0]:.0f} .. {agg.tof_edges[-1]:.0f}, unit 100 ps)")
print(f"shots / bin : {agg.n_per_bin}")
print(f"G per bin   : {np.round(agg.G, 3)}")
print(f"Var(G)/bin  : {np.round(agg.var_G, 4)}")

## TOF -> KE calibration

Load the Task-1 calibration `KE(t) = (b / (t - t0))^2 + V_ret` and map the
eTOF bin edges onto KE. Only the contiguous span of bins with `t > t0` and
centre KE inside `KE_PLOT_RANGE` is kept (KE is monotone decreasing in TOF
there, so the span is a single block and the KE edge array stays monotone —
exactly what `pcolormesh` needs).

In [ ]:
with open(CALIBRATION_JSON) as fh:
    calib = json.load(fh)
t0_ns = float(calib["t0_ns"])
b_cal = float(calib["b"])
v_ret = float(calib["retarding_voltage_V"])
print(f"calibration : {CALIBRATION_JSON.name}")
print(f"model       : {calib['model']}")
print(f"t0 = {t0_ns:.3f} ns, b = {b_cal:.3f}, V_ret = {v_ret:g} V")
print(f"(rms {calib['rms_tof_residual_ns']:.3f} ns, runs {calib['run_ar']}/{calib['run_bg']})")

def tof_to_ke(t_ns):
    """Born electron KE (eV) for a TOF in ns; NaN for t <= t0."""
    t_ns = np.asarray(t_ns, dtype=np.float64)
    return np.where(t_ns > t0_ns, (b_cal / (t_ns - t0_ns)) ** 2 + v_ret, np.nan)

# eTOF bin geometry in ns and KE.
tof_edges_ns = np.asarray(agg.tof_edges, dtype=np.float64) * TOF_UNIT_NS
tof_cent_ns  = 0.5 * (tof_edges_ns[:-1] + tof_edges_ns[1:])
ke_edges_all = tof_to_ke(tof_edges_ns)     # (n_tof + 1,), decreasing where valid
ke_cent_all  = tof_to_ke(tof_cent_ns)      # (n_tof,)

# Contiguous span of usable bins: both edges finite, centre inside range.
edge_ok = np.isfinite(ke_edges_all[:-1]) & np.isfinite(ke_edges_all[1:])
good = (edge_ok & np.isfinite(ke_cent_all)
        & (ke_cent_all >= KE_PLOT_RANGE[0]) & (ke_cent_all <= KE_PLOT_RANGE[1]))
idx = np.flatnonzero(good)
assert idx.size > 0, "no eTOF bins map into KE_PLOT_RANGE - check units / calibration"
assert np.all(np.diff(idx) == 1), "selected KE bins are not contiguous - check calibration"
sel = slice(idx[0], idx[-1] + 1)

ke_cent  = ke_cent_all[sel]                    # (n_sel,)
ke_edges = ke_edges_all[idx[0] : idx[-1] + 2]  # (n_sel + 1,), monotone decreasing
ke_width = np.abs(np.diff(ke_edges))           # eV covered by each TOF bin
n_sel = ke_cent.size
print(f"{n_sel} of {agg.n_tof} eTOF bins map into {KE_PLOT_RANGE} eV")
print(f"(TOF {tof_edges_ns[idx[0]]:.0f} .. {tof_edges_ns[idx[-1] + 1]:.0f} ns)")

## Covariance and the GMD-driven correction

Per GMD bin (all aggregate datasets are per-bin means):

    Cov(D, D) = <D D^T> - <D><D>^T                      (n_sel, n_sel)
    Cov(D, G) = <D G>   - <D><G>                        (n_sel,)
    Corr      = Cov(D, G) Cov(D, G)^T / Var(G)          rank-1, always PSD

`Corr` is the map an entirely GMD-driven (linear common-mode) correlation
would produce; `alpha` scales how much of it is subtracted.

In [ ]:
# Electron-electron covariance per GMD bin, on the full TOF grid.
cov_DD_full = agg.DtD - agg.D[..., :, None] * agg.D[..., None, :]
cov_DG_full = agg.DtG - agg.D * agg.G[..., None]

var_G = agg.var_G
safe_var = np.where(var_G > 0, var_G, np.nan)

# Rank-1 GMD-driven common-mode term (this is what alpha scales).
corr_DD_full = (cov_DG_full[..., :, None] * cov_DG_full[..., None, :]) / safe_var[..., None, None]

# Restrict to the KE-mapped span (sel is a slice, so this is basic slicing).
cov_DD  = cov_DD_full[:, sel, sel]    # (n_gmd, n_sel, n_sel)
corr_DD = corr_DD_full[:, sel, sel]
print(f"restricted covariance shape: {cov_DD.shape}")

## A-priori-uncorrelated region and the optimal scale `alpha*`

Build the boolean region mask from the KE rectangles (symmetrised, diagonal
ridge removed), then solve the 1-parameter least-squares problem per GMD bin.
Note: `alpha*` is computed on the raw TOF-bin covariances (uniform TOF-bin
weighting); the Jacobian correction below is applied only for display, and a
per-eV^2 weighting would shift `alpha*` slightly if the region spans very
different KE.

In [ ]:
# Region mask over the (n_sel, n_sel) map.
roi_mask = np.zeros((n_sel, n_sel), dtype=bool)
for (a_lo, a_hi), (b_lo, b_hi) in UNCORRELATED_ROI_KE:
    in_a = (ke_cent >= a_lo) & (ke_cent <= a_hi)
    in_b = (ke_cent >= b_lo) & (ke_cent <= b_hi)
    roi_mask |= np.outer(in_a, in_b)
roi_mask |= roi_mask.T                        # Cov(D, D) is symmetric
near_diag = np.abs(ke_cent[:, None] - ke_cent[None, :]) < DIAG_EXCLUDE_EV
roi_mask &= ~near_diag
print(f"region: {roi_mask.sum()} of {n_sel * n_sel} map elements")

def roi_norm(M):
    """Frobenius norm of a map restricted to the uncorrelated region."""
    return float(np.sqrt(np.nansum(M[roi_mask] ** 2)))

# Closed-form least-squares scale per GMD bin:
# alpha* = sum_R(Cov * Corr) / sum_R(Corr^2).
alpha_star = np.full(agg.n_gmd_bins, np.nan)
for bn in range(agg.n_gmd_bins):
    if agg.n_per_bin[bn] == 0:
        continue
    c = cov_DD[bn][roi_mask]
    k = corr_DD[bn][roi_mask]
    denom = np.nansum(k * k)
    if denom > 0:
        alpha_star[bn] = float(np.nansum(c * k) / denom)

print()
print(f"{'bin':>3}  {'n':>8}  {'<G>':>7}  {'alpha*':>8}  "
      f"{'|res(0)|':>10}  {'|res(1)|':>10}  {'|res(a*)|':>10}")
for bn in range(agg.n_gmd_bins):
    if agg.n_per_bin[bn] == 0 or not np.isfinite(alpha_star[bn]):
        continue
    r0 = roi_norm(cov_DD[bn])
    r1 = roi_norm(cov_DD[bn] - corr_DD[bn])
    rs = roi_norm(cov_DD[bn] - alpha_star[bn] * corr_DD[bn])
    print(f"{bn:>3}  {int(agg.n_per_bin[bn]):>8d}  {agg.G[bn]:>7.3f}  "
          f"{alpha_star[bn]:>8.3f}  {r0:>10.3e}  {r1:>10.3e}  {rs:>10.3e}")

In [ ]:
# Residual-in-region vs alpha, one curve per GMD bin. alpha* (dotted) should
# sit at each curve's minimum; the dashed line is the standard partial.
fig, ax = plt.subplots(figsize=(7.5, 5), constrained_layout=True)
for bn in range(agg.n_gmd_bins):
    if agg.n_per_bin[bn] == 0 or not np.isfinite(alpha_star[bn]):
        continue
    res = [roi_norm(cov_DD[bn] - a * corr_DD[bn]) for a in ALPHA_SCAN]
    line, = ax.plot(ALPHA_SCAN, res, lw=1,
                    label=f"bin {bn} (G={agg.G[bn]:.2f}, a*={alpha_star[bn]:.2f})")
    ax.axvline(alpha_star[bn], color=line.get_color(), lw=0.6, ls=":")
ax.axvline(1.0, color="k", lw=0.8, ls="--", label="standard partial (alpha = 1)")
ax.set_xlabel("alpha (common-mode subtraction scale)")
ax.set_ylabel("residual norm in uncorrelated region")
ax.set_yscale("log")
ax.legend(fontsize=8)
ax.set_title("Scaling the GMD common-mode subtraction")
plt.show()

## Covariance maps on the KE axis

One panel per GMD bin; both axes are electron KE from the Task-1 calibration.
With `JACOBIAN_CORRECT = True` the maps are densities per eV^2
(`Cov / (dKE_i dKE_j)`), which undoes the apparent enhancement of slow
electrons that the compressed low-KE end of the TOF -> KE mapping would
otherwise produce. Dashed rectangles mark the a-priori-uncorrelated region
used to fit `alpha*`.

In [ ]:
def _to_density(M):
    """Convert a covariance on the TOF-bin grid to a per-eV^2 density."""
    if not JACOBIAN_CORRECT:
        return M
    return M / (ke_width[:, None] * ke_width[None, :])

def _grid(n):
    cols = min(4, n)
    return (n + cols - 1) // cols, cols

def _draw_roi(ax):
    for (a_lo, a_hi), (b_lo, b_hi) in UNCORRELATED_ROI_KE:
        for (x_lo, x_hi), (y_lo, y_hi) in (((a_lo, a_hi), (b_lo, b_hi)),
                                           ((b_lo, b_hi), (a_lo, a_hi))):
            ax.add_patch(plt.Rectangle((x_lo, y_lo), x_hi - x_lo, y_hi - y_lo,
                                       fill=False, edgecolor="k", lw=0.8, ls="--"))

def plot_cov_ke_grid(matrices, title, show_roi=True):
    """One covariance panel per GMD bin on KE axes (eV)."""
    rows, cols = _grid(agg.n_gmd_bins)
    fig, axes = plt.subplots(rows, cols, figsize=(4.6 * cols, 3.9 * rows))
    axes_flat = np.atleast_1d(axes).ravel()

    for bn in range(agg.n_gmd_bins):
        ax = axes_flat[bn]
        M = _to_density(matrices[bn])
        if not np.isfinite(M).any():
            ax.set_title(f"bin {bn} (empty)")
            ax.axis("off")
            continue
        vmax = float(np.nanpercentile(np.abs(M), CLIM_PERCENTILE)) or 1.0
        pm = ax.pcolormesh(ke_edges, ke_edges, M, cmap="RdBu_r",
                           vmin=-vmax, vmax=vmax, shading="flat")
        ax.set_xlim(KE_PLOT_RANGE)
        ax.set_ylim(KE_PLOT_RANGE)
        if show_roi:
            _draw_roi(ax)
        ax.set_title(f"GMD [{agg.gmd_edges[bn]:.2f}, {agg.gmd_edges[bn + 1]:.2f}) uJ"
                     f"   n={int(agg.n_per_bin[bn])}", fontsize=10)
        ax.set_xlabel("electron KE (eV)")
        ax.set_ylabel("electron KE (eV)")
        fig.colorbar(pm, ax=ax, fraction=0.046)

    for ax in axes_flat[agg.n_gmd_bins:]:
        ax.axis("off")
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    plt.show()

plot_cov_ke_grid(cov_DD, "Cov(D, D) on the KE axis - raw")

In [ ]:
# Standard partial covariance: full unit-weight subtraction (alpha = 1).
plot_cov_ke_grid(cov_DD - corr_DD, "pCov(alpha = 1) - standard partial covariance")

In [ ]:
# Region-minimising subtraction: per-bin alpha* from the closed form above.
scaled = cov_DD - alpha_star[:, None, None] * corr_DD
plot_cov_ke_grid(scaled, "pCov(alpha = alpha*) - region-minimising scale")

## Single-bin comparison

Side-by-side for `FOCUS_BIN`: raw, `alpha = 1`, `alpha = alpha*`, and the map
of what changed between the two subtractions,
`pCov(alpha*) - pCov(1) = (1 - alpha*) Corr`. The first three panels share a
colour scale (set from the `alpha = 1` map) so shape changes are directly
comparable; the difference panel has its own scale.

In [ ]:
bn = FOCUS_BIN % agg.n_gmd_bins
assert agg.n_per_bin[bn] > 0 and np.isfinite(alpha_star[bn]), "focus bin is empty"

panels = [
    (cov_DD[bn],                                "raw Cov(D, D)"),
    (cov_DD[bn] - corr_DD[bn],                  "pCov(alpha = 1)"),
    (cov_DD[bn] - alpha_star[bn] * corr_DD[bn], f"pCov(alpha* = {alpha_star[bn]:.3f})"),
    ((1.0 - alpha_star[bn]) * corr_DD[bn],      "pCov(alpha*) - pCov(1)"),
]

ref = _to_density(cov_DD[bn] - corr_DD[bn])
vmax_shared = float(np.nanpercentile(np.abs(ref), CLIM_PERCENTILE)) or 1.0

fig, axes = plt.subplots(1, 4, figsize=(20, 4.4))
for i, (ax, (M, ttl)) in enumerate(zip(axes, panels)):
    Md = _to_density(M)
    if i < 3:
        vmax = vmax_shared
    else:
        vmax = float(np.nanpercentile(np.abs(Md), CLIM_PERCENTILE)) or 1.0
    pm = ax.pcolormesh(ke_edges, ke_edges, Md, cmap="RdBu_r",
                       vmin=-vmax, vmax=vmax, shading="flat")
    ax.set_xlim(KE_PLOT_RANGE)
    ax.set_ylim(KE_PLOT_RANGE)
    _draw_roi(ax)
    ax.set_title(ttl, fontsize=10)
    ax.set_xlabel("electron KE (eV)")
    if i == 0:
        ax.set_ylabel("electron KE (eV)")
    fig.colorbar(pm, ax=ax, fraction=0.046)

fig.suptitle(f"GMD bin {bn}: [{agg.gmd_edges[bn]:.2f}, {agg.gmd_edges[bn + 1]:.2f}) uJ, "
             f"n={int(agg.n_per_bin[bn])}", y=1.03)
fig.tight_layout()
plt.show()

## Notes / caveats

- **alpha* is only as good as the region.** It must be free of real physics
  (no photoline-Auger or Auger-Auger coincidences between the two KE ranges)
  yet still carry visible common mode in the raw map; pick it from the raw
  panels above and re-run. Several disjoint rectangle pairs can be listed in
  `UNCORRELATED_ROI_KE` — they are OR-combined.
- `alpha* > 1` means the standard partial under-subtracts in the region
  (residual common mode, e.g. a nonlinear GMD response); `alpha* < 1` means
  it over-subtracts (part of the "common mode" there is real correlated
  signal leaking into the rank-1 estimate).
- The subtraction only changes the map along the rank-1 direction
  `Cov(D, G) Cov(D, G)^T`; genuinely two-particle features orthogonal to it
  are untouched by any alpha.
- `alpha*` is fitted on the TOF-bin covariances (uniform bin weighting); the
  Jacobian correction is display-only.
- The same machinery applies verbatim to `Cov(C, C)` (ion-ion) and
  `Cov(D, C)` (electron-ion) — swap the aggregate fields as in
  `covariance_tests.ipynb`.